## 1. Import Necessary Libraries

In [47]:
import pandas as pd
from mlxtend.frequent_patterns import apriori,association_rules
import warnings
warnings.filterwarnings('ignore')

## 2. Data Collection

In [3]:
online_retail_data=pd.read_excel('Online Retail.xlsx')
online_retail_data

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France


## 3. Data Understanding

In [4]:
online_retail_data.shape

(541909, 8)

In [5]:
online_retail_data.isna().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [7]:
online_retail_data.describe(include='all')

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
count,541909.0,541909,540455,541909.000000,541909,541909.000000,406829.000000,541909
unique,25900.0,4070,4223,NaN,NaN,NaN,NaN,38
top,573585.0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,NaN,NaN,NaN,NaN,United Kingdom
freq,1114.0,2313,2369,NaN,NaN,NaN,NaN,495478
mean,NaN,NaN,NaN,9.552250,2011-07-04 13:34:57.156386,4.611114,15287.690570,NaN
min,NaN,NaN,NaN,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000,NaN
25%,NaN,NaN,NaN,1.000000,2011-03-28 11:34:00,1.250000,13953.000000,NaN
50%,NaN,NaN,NaN,3.000000,2011-07-19 17:17:00,2.080000,15152.000000,NaN
75%,NaN,NaN,NaN,10.000000,2011-10-19 11:27:00,4.130000,16791.000000,NaN
max,NaN,NaN,NaN,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000,NaN


In [8]:
online_retail_data.dtypes

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object

## 4. Data Preparation

In [9]:
online_retail_data.isna().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [10]:
# the contribution of 'CustomerID' not important at here, so I removing it
# then, to generate frequent set-items we need the products, without the product we unable to do that process, so better we drop that row,
# because we cannot randomly impute values to it

In [12]:
1454/541909 # the percent of missing values in the Description, so low and proceed our plan

0.002683107311375157

In [13]:
del online_retail_data['CustomerID']

In [14]:
online_retail_data.dropna(inplace=True)

In [15]:
online_retail_data.isna().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
Country        0
dtype: int64

In [16]:
# now the dataset is clean

## 5. Data Mining

In [19]:
online_retail_data.describe(include='all')

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,Country
count,540455.0,540455,540455,540455.000000,540455,540455.000000,540455
unique,24446.0,3958,4223,NaN,NaN,NaN,38
top,573585.0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,NaN,NaN,NaN,United Kingdom
freq,1114.0,2313,2369,NaN,NaN,NaN,494024
mean,NaN,NaN,NaN,9.603129,2011-07-04 16:20:42.947035,4.623519,NaN
min,NaN,NaN,NaN,-80995.000000,2010-12-01 08:26:00,-11062.060000,NaN
25%,NaN,NaN,NaN,1.000000,2011-03-28 11:49:00,1.250000,NaN
50%,NaN,NaN,NaN,3.000000,2011-07-20 11:38:00,2.080000,NaN
75%,NaN,NaN,NaN,10.000000,2011-10-19 11:49:00,4.130000,NaN
max,NaN,NaN,NaN,80995.000000,2011-12-09 12:50:00,38970.000000,NaN


### there are 38 countries, since we already got the best association rules for UK, let's do it for next country

In [20]:
online_retail_data['Country'].unique()

<ArrowStringArray>
[      'United Kingdom',               'France',            'Australia',
          'Netherlands',              'Germany',               'Norway',
                 'EIRE',          'Switzerland',                'Spain',
               'Poland',             'Portugal',                'Italy',
              'Belgium',            'Lithuania',                'Japan',
              'Iceland',      'Channel Islands',              'Denmark',
               'Cyprus',               'Sweden',              'Austria',
               'Israel',              'Finland',              'Bahrain',
               'Greece',            'Hong Kong',            'Singapore',
              'Lebanon', 'United Arab Emirates',         'Saudi Arabia',
       'Czech Republic',               'Canada',          'Unspecified',
               'Brazil',                  'USA',   'European Community',
                'Malta',                  'RSA']
Length: 38, dtype: str

In [22]:
online_retail_data['Country'].value_counts().head()

Country
United Kingdom    494024
Germany             9495
France              8557
EIRE                8196
Spain               2533
Name: count, dtype: int64

### Find the best assiocation rules for Germany

In [24]:
germany_data = online_retail_data[online_retail_data['Country']=='Germany']
germany_data

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,Country
1109,536527,22809,SET OF 6 T-LIGHTS SANTA,6,2010-12-01 13:04:00,2.95,Germany
1110,536527,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,6,2010-12-01 13:04:00,2.55,Germany
1111,536527,84945,MULTI COLOUR SILVER T-LIGHT HOLDER,12,2010-12-01 13:04:00,0.85,Germany
1112,536527,22242,5 HOOK HANGER MAGIC TOADSTOOL,12,2010-12-01 13:04:00,1.65,Germany
1113,536527,22244,3 HOOK HANGER MAGIC GARDEN,12,2010-12-01 13:04:00,1.95,Germany
...,...,...,...,...,...,...,...
541801,581578,22993,SET OF 4 PANTRY JELLY MOULDS,12,2011-12-09 12:16:00,1.25,Germany
541802,581578,22907,PACK OF 20 NAPKINS PANTRY DESIGN,12,2011-12-09 12:16:00,0.85,Germany
541803,581578,22908,PACK OF 20 NAPKINS RED APPLES,12,2011-12-09 12:16:00,0.85,Germany
541804,581578,23215,JINGLE BELL HEART ANTIQUE SILVER,12,2011-12-09 12:16:00,2.08,Germany


In [25]:
germany_data.isna().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
Country        0
dtype: int64

In [26]:
# to find the best association rule, we need to get the best frequent set-items
# so, we need the one-hot df the fill with transaction(row),product(columns),value(quantity)

In [28]:
germany_pivotted_data=pd.pivot_table(data=germany_data,values='Quantity',index='InvoiceNo',columns='Description',aggfunc='sum').fillna(0)
germany_pivotted_data

Description,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 IVORY ROSE PEG PLACE SETTINGS,12 MESSAGE CARDS WITH ENVELOPES,...,YULETIDE IMAGES GIFT WRAP SET,ZINC HEART T-LIGHT HOLDER,ZINC STAR T-LIGHT HOLDER,ZINC BOX SIGN HOME,ZINC FOLKART SLEIGH BELLS,ZINC HEART LATTICE T-LIGHT HOLDER,ZINC METAL HEART DECORATION,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC WILLIE WINKIE CANDLE STICK
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536527,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536840,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536861,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536967,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536983,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
C580313,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
C580714,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
C580740,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [29]:
# based on the pivot table :
# we can see that - 603 transactions happenedt
# there are 1703 unique product selling in germany

### Now, we got the pivot table of the information we needed - so, now convert into one-hot df

In [30]:
germany_onehot_data=germany_pivotted_data.map(lambda x:0  if x==0 else 1)
germany_onehot_data

Description,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 IVORY ROSE PEG PLACE SETTINGS,12 MESSAGE CARDS WITH ENVELOPES,...,YULETIDE IMAGES GIFT WRAP SET,ZINC HEART T-LIGHT HOLDER,ZINC STAR T-LIGHT HOLDER,ZINC BOX SIGN HOME,ZINC FOLKART SLEIGH BELLS,ZINC HEART LATTICE T-LIGHT HOLDER,ZINC METAL HEART DECORATION,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC WILLIE WINKIE CANDLE STICK
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536527,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536840,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536861,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536967,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536983,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
C580313,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
C580714,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
C580740,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 7. Building Frequent Set-Items || 8. Building Association Rules

In [36]:
frequent_set_items=apriori(df=germany_onehot_data,min_support=0.03,use_colnames=True)
frequent_set_items

,support,itemsets
0,0.033167,frozenset({3 PIECE SPACEBOY COOKIE CUTTER SET})
1,0.082919,frozenset({6 RIBBONS RUSTIC CHARM})
2,0.054726,frozenset({ALARM CLOCK BAKELIKE PINK})
3,0.034826,frozenset({ALARM CLOCK BAKELIKE RED })
4,0.038143,frozenset({BAKING SET 9 PIECE RETROSPOT })
...,...,...
177,0.094527,"frozenset({POSTAGE, ROUND SNACK BOXES SET OF 4..."
178,0.031509,"frozenset({POSTAGE, ROUND SNACK BOXES SET OF 4..."
179,0.046434,"frozenset({SPACEBOY LUNCH BOX , POSTAGE, ROUND..."
180,0.031509,"frozenset({STRAWBERRY LUNCH BOX WITH CUTLERY, ..."


In [40]:
best_association_germany=association_rules(df=frequent_set_items,metric='confidence',min_threshold=0.3)
best_association_germany

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({6 RIBBONS RUSTIC CHARM}),frozenset({POSTAGE}),0.082919,0.635158,0.069652,0.840000,1.322507,1.0,0.016985,2.280265,0.265909,0.107417,0.561455,0.474830
1,frozenset({6 RIBBONS RUSTIC CHARM}),frozenset({REGENCY CAKESTAND 3 TIER}),0.082919,0.134328,0.033167,0.400000,2.977778,1.0,0.022029,1.442786,0.724231,0.180180,0.306897,0.323457
2,frozenset({ALARM CLOCK BAKELIKE PINK}),frozenset({POSTAGE}),0.054726,0.635158,0.038143,0.696970,1.097318,1.0,0.003383,1.203980,0.093822,0.058524,0.169421,0.378511
3,frozenset({BAKING SET 9 PIECE RETROSPOT }),frozenset({POSTAGE}),0.038143,0.635158,0.033167,0.869565,1.369054,1.0,0.008941,2.797125,0.280259,0.051813,0.642490,0.460892
4,frozenset({BLUE HARMONICA IN BOX }),frozenset({POSTAGE}),0.036484,0.635158,0.036484,1.000000,1.574413,1.0,0.013311,inf,0.378657,0.057441,1.000000,0.528721
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159,"frozenset({STRAWBERRY LUNCH BOX WITH CUTLERY, ...",frozenset({POSTAGE}),0.034826,0.635158,0.031509,0.904762,1.424468,1.0,0.009389,3.830846,0.308736,0.049351,0.738961,0.477185
160,frozenset({STRAWBERRY LUNCH BOX WITH CUTLERY}),"frozenset({POSTAGE, ROUND SNACK BOXES SET OF4 ...",0.063018,0.170813,0.031509,0.500000,2.927184,1.0,0.020745,1.658375,0.702655,0.155738,0.397000,0.342233
161,"frozenset({POSTAGE, WOODLAND CHARLOTTE BAG})",frozenset({ROUND SNACK BOXES SET OF4 WOODLAND }),0.087894,0.197347,0.044776,0.509434,2.581417,1.0,0.027431,1.636178,0.671650,0.186207,0.388820,0.368162
162,"frozenset({WOODLAND CHARLOTTE BAG, ROUND SNACK...",frozenset({POSTAGE}),0.048093,0.635158,0.044776,0.931034,1.465832,1.0,0.014230,5.290216,0.333850,0.070130,0.810972,0.500765


In [49]:
best_association_germany.dtypes

antecedents            object
consequents            object
antecedent support    float64
consequent support    float64
support               float64
confidence            float64
lift                  float64
representativity      float64
leverage              float64
conviction            float64
zhangs_metric         float64
jaccard               float64
certainty             float64
kulczynski            float64
dtype: object

In [50]:
best_association_germany['antecedents'].str.replace(r'frozenset\(\{','',regex=True).str.replace(r'\)\}','',regex=True)

AttributeError: Can only use .str accessor with string values, not floating

In [ ]:
best_association_germany['antecedents'].

In [46]:
best_association_germany

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({6 RIBBONS RUSTIC CHARM}),frozenset({POSTAGE}),0.082919,0.635158,0.069652,0.840000,1.322507,1.0,0.016985,2.280265,0.265909,0.107417,0.561455,0.474830
1,frozenset({6 RIBBONS RUSTIC CHARM}),frozenset({REGENCY CAKESTAND 3 TIER}),0.082919,0.134328,0.033167,0.400000,2.977778,1.0,0.022029,1.442786,0.724231,0.180180,0.306897,0.323457
2,frozenset({ALARM CLOCK BAKELIKE PINK}),frozenset({POSTAGE}),0.054726,0.635158,0.038143,0.696970,1.097318,1.0,0.003383,1.203980,0.093822,0.058524,0.169421,0.378511
3,frozenset({BAKING SET 9 PIECE RETROSPOT }),frozenset({POSTAGE}),0.038143,0.635158,0.033167,0.869565,1.369054,1.0,0.008941,2.797125,0.280259,0.051813,0.642490,0.460892
4,frozenset({BLUE HARMONICA IN BOX }),frozenset({POSTAGE}),0.036484,0.635158,0.036484,1.000000,1.574413,1.0,0.013311,inf,0.378657,0.057441,1.000000,0.528721
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159,"frozenset({STRAWBERRY LUNCH BOX WITH CUTLERY, ...",frozenset({POSTAGE}),0.034826,0.635158,0.031509,0.904762,1.424468,1.0,0.009389,3.830846,0.308736,0.049351,0.738961,0.477185
160,frozenset({STRAWBERRY LUNCH BOX WITH CUTLERY}),"frozenset({POSTAGE, ROUND SNACK BOXES SET OF4 ...",0.063018,0.170813,0.031509,0.500000,2.927184,1.0,0.020745,1.658375,0.702655,0.155738,0.397000,0.342233
161,"frozenset({POSTAGE, WOODLAND CHARLOTTE BAG})",frozenset({ROUND SNACK BOXES SET OF4 WOODLAND }),0.087894,0.197347,0.044776,0.509434,2.581417,1.0,0.027431,1.636178,0.671650,0.186207,0.388820,0.368162
162,"frozenset({WOODLAND CHARLOTTE BAG, ROUND SNACK...",frozenset({POSTAGE}),0.048093,0.635158,0.044776,0.931034,1.465832,1.0,0.014230,5.290216,0.333850,0.070130,0.810972,0.500765


In [43]:
best_association_germany.to_csv(r'D:\Management\upm\SEM8\Paid_Data_Science\An Important_Syllabus\Machine Learning\CHAPTER20-ASSOCIATION RULES\germany_best_association_rules.csv')

## THE END

In [51]:
# Need to look into cleaning the frozenset wording